# Paid to Disagree: cross-model round-robin GRPO (C7)

Trains the council (Qwen2.5-1.5B, Llama-3.2-1B, Gemma-3-1B) one member at a time against the
other two, frozen, with the peer-prediction reward (`configs/train.yaml`, `configs/reward.yaml`).
One **leg** = one member trained for `grpo.max_steps` steps; 2 rounds x 3 members = 6 legs. A
Kaggle commit is limited to 12 hours, so the notebook runs **one leg per session** and keeps all
progress in `runs/cross_model_v1/` (`state.json` says which leg is next).

**Notebook settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
Optional: Add-ons -> Secrets -> `HF_TOKEN` (after accepting the Llama 3.2 and Gemma 3 licences);
without it the notebook uses Unsloth's ungated copies of the same weights.

**First session:** keep `MODE = "pilot"` and run all cells interactively (about 30-45 min). The
pilot caches all three members on 40 questions and trains Qwen for 20 steps. It checks that
Unsloth / TRL / vLLM work on the T4 and measures seconds per step.

**Every later session:** set `MODE = "leg"`. From the second leg on, attach the previous
session's `c7_runs.zip` output as a dataset (Add Input) and set `RESUME_FROM` to its folder.
Then **Save Version -> Save & Run All (Commit)**. Each session restores `runs/`, refreshes any
stale peer cache, trains the next leg, re-caches the member it trained, and zips `runs/` as
`c7_runs.zip`.

In [ ]:
# ---- settings ----
REPO = "https://github.com/nithish-74/paid-to-disagree.git"
CONFIG = "configs/train.yaml"
MODE = "pilot"             # "pilot" (first session) or "leg" (one training leg per commit)
RESUME_FROM = None         # e.g. "/kaggle/input/c7-runs/runs" (unzipped c7_runs.zip of the last session)
MAX_STEPS = None           # None = configs/train.yaml (800); lower it if the pilot says a leg won't fit in ~10 h
PILOT = dict(max_train_questions=40, max_steps=20, monitor_every=10, dev_n=20)

## 1. Code and packages

In [ ]:
import os, subprocess, json, time, pathlib, shutil
os.chdir("/kaggle/working")
if not pathlib.Path("ptd").exists():
    subprocess.run(["git", "clone", "-q", REPO, "ptd"], check=True)
os.chdir("/kaggle/working/ptd")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)  # pick up fixes when re-running
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Unsloth first (it pins compatible torch / transformers / trl), then vLLM for fast generation.
!pip install -q unsloth
!pip install -q vllm trl
!pip install -q -e .
!python -c "import unsloth, trl, vllm, torch; print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 2. Model access and configs

In [ ]:
import yaml

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

council = yaml.safe_load(open("configs/council.yaml"))
if token:
    os.environ["HF_TOKEN"] = token
    print("Using the official gated models with your HF_TOKEN secret.")
else:
    mirrors = {"meta-llama/Llama-3.2-1B-Instruct": "unsloth/Llama-3.2-1B-Instruct",
               "google/gemma-3-1b-it": "unsloth/gemma-3-1b-it"}
    for m in council["members"]:
        m["backend"]["model"] = mirrors.get(m["backend"]["model"], m["backend"]["model"])
    print("No HF_TOKEN secret: using Unsloth's ungated copies of the same weights.")
yaml.safe_dump(council, open("configs/council_kaggle.yaml", "w"), sort_keys=False)

cfg = yaml.safe_load(open(CONFIG))
cfg["council_config"] = "configs/council_kaggle.yaml"
if MAX_STEPS:
    cfg["grpo"]["max_steps"] = MAX_STEPS
yaml.safe_dump(cfg, open("configs/train_kaggle.yaml", "w"), sort_keys=False)

pilot = {**cfg, "experiment": cfg["experiment"] + "_pilot", "max_train_questions": PILOT["max_train_questions"]}
pilot["grpo"] = {**cfg["grpo"], "max_steps": PILOT["max_steps"]}
pilot["monitor"] = {**cfg["monitor"], "every": PILOT["monitor_every"], "dev_n": PILOT["dev_n"]}
yaml.safe_dump(pilot, open("configs/train_kaggle_pilot.yaml", "w"), sort_keys=False)
print("experiment:", cfg["experiment"], "| steps per leg:", cfg["grpo"]["max_steps"], "| rotation:", cfg["rotation"])

## 3. Question data (training sources only)

In [ ]:
!python -m data.build_questions --config configs/data.yaml --sources truthfulqa_mc1 arc_challenge mmlu_pro
!wc -l data/processed/*.jsonl

## 4. Helpers

Each command runs in its own process (frees GPU memory between cache and train) and logs to
`runs/<experiment>.<command>.log`; the last line is printed every few minutes.

In [ ]:
def run_step(command, config, *extra, every=120, tail=15):
    exp = yaml.safe_load(open(config))["experiment"]
    log_path = pathlib.Path(f"runs/{exp}.{command}.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    start = time.time()
    with open(log_path, "a") as log:
        p = subprocess.Popen(["python", "-m", "train.grpo_roundrobin", command, "--config", config, *extra],
                             stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        while p.poll() is None:
            time.sleep(every)
            lines = [l.rstrip() for l in open(log_path, errors="replace") if l.strip()]
            print(f"[{(time.time() - start) / 60:5.1f} min] {command}: {lines[-1][:200] if lines else '...'}", flush=True)
    lines = [l.rstrip() for l in open(log_path, errors="replace") if l.strip()]
    print("\n".join(lines[-tail:]))
    print(f"{command} {' '.join(extra)} exit code {p.returncode} after {(time.time() - start) / 60:.1f} min")
    assert p.returncode == 0, f"{command} failed: see {log_path}"
    return time.time() - start


def status(config):
    out = subprocess.run(["python", "-m", "train.grpo_roundrobin", "status", "--config", config],
                         capture_output=True, text=True).stdout
    print(out)
    return out


def state_of(config):
    exp = yaml.safe_load(open(config))["experiment"]
    p = pathlib.Path("runs") / exp / "state.json"
    return json.load(open(p)) if p.exists() else {}


def show_legs(config):
    exp = pathlib.Path("runs") / yaml.safe_load(open(config))["experiment"]
    for leg_dir in sorted(exp.glob("round*_*")):
        print(f"--- {leg_dir}")
        for name in ("monitor.jsonl", "reward_terms.jsonl"):
            f = leg_dir / name
            if f.exists():
                rows = [json.loads(l) for l in open(f) if l.strip()]
                print(f"{name}: {len(rows)} lines")
                for r in rows[:2] + rows[-2:]:
                    print("  ", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items() if k != "history"})

## 5. Pilot (MODE = "pilot"): 40 questions, cache all members, 20 steps for Qwen

Same code path as a real leg at toy size. Passes if every step exits cleanly and
`reward_terms.jsonl` shows every reward term. Seconds per step sets how many steps a real leg
can afford in one session (keep a leg under ~10 h to leave room for caching).

In [ ]:
if MODE == "pilot":
    P = "configs/train_kaggle_pilot.yaml"
    shutil.rmtree(f"runs/{yaml.safe_load(open(P))['experiment']}", ignore_errors=True)  # a re-run starts clean
    t_cache = run_step("cache", P, every=60)
    t_train = run_step("train", P, every=60)
    show_legs(P)
    status(P)
    steps = PILOT["max_steps"]
    print(f"pilot cache: {t_cache / 60:.1f} min for 3 members x {PILOT['max_train_questions']} questions")
    print(f"pilot train: {t_train / 60:.1f} min for {steps} steps incl. model load and dev checks "
          f"-> about {t_train / steps:.0f} s/step as an upper bound")
    full_q = cfg["max_train_questions"]
    print(f"full cache estimate: {t_cache / 60 * full_q / PILOT['max_train_questions']:.0f} min "
          f"({full_q} questions); full leg estimate: {t_train / steps * cfg['grpo']['max_steps'] / 3600:.1f} h "
          f"({cfg['grpo']['max_steps']} steps)")
else:
    print("MODE is not 'pilot': skipping the pilot")

## 6. One training leg (MODE = "leg")

Restores `runs/` from `RESUME_FROM`, caches any member whose cache is missing or stale, trains
the next leg, then re-caches the member it just trained so the next leg sees its new answers.

In [ ]:
if MODE == "leg":
    C = "configs/train_kaggle.yaml"
    if RESUME_FROM:
        src = pathlib.Path(RESUME_FROM)
        assert src.exists(), f"{src} not found: attach the last c7_runs.zip as a dataset"
        shutil.copytree(src, "runs", dirs_exist_ok=True)
        print("restored runs/ from", src)
    status(C)
    st = state_of(C)
    missing = [m for m in cfg["rotation"] if m not in st.get("cache", {})]
    stale = [m for m in cfg["rotation"] if m in st.get("cache", {}) and st["cache"][m] != st.get("loras", {}).get(m)]
    if missing or stale:
        run_step("cache", C, "--members", *(missing + stale), every=300)
    before = dict(state_of(C).get("loras", {}))
    run_step("train", C, every=600)
    after = state_of(C).get("loras", {})
    trained = [m for m in after if after[m] != before.get(m)]
    if trained:
        run_step("cache", C, "--members", *trained, every=300)
    show_legs(C)
    status(C)
else:
    print("MODE is not 'leg': skipping training")

## 7. Package the results (attach this zip as a dataset for the next session)

In [ ]:
out = shutil.make_archive("/kaggle/working/c7_runs", "zip", ".", "runs")
print(out, f"{os.path.getsize(out) / 1e6:.1f} MB")